# Improved Grid Defense — Ablation Test

**Purpose:** raise multilingual grid-defense accuracy above the current ~11% mean by
ablating black-box improvements (families A–D). Runs on the **100-image tune subset**.

| Family | What |
|---|---|
| baseline | original 4×4 max-conf + mean-fill (1p / greedy 2p) |
| B | conf-drop of pre-defense top class (+ EN–ZH agree bonus) |
| A | hierarchical refine; 6×6; overlapping stride-28 |
| C | Gaussian blur occlusion vs mean-fill |
| D | distance-aware 2nd patch + beam-2 |

Results → `results/comparison.json`. Promote to full 1000 only if mean acc ≥ ~20% at cost ≤ ~80.


In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'open_clip_torch', 'transformers', 'datasets', 'matplotlib', 'Pillow', 'scipy'], check=False)


CompletedProcess(args=['D:\\ian\\2026summer\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'open_clip_torch', 'transformers', 'datasets', 'matplotlib', 'Pillow', 'scipy'], returncode=0)

In [2]:
import os, random, json, time
from itertools import combinations
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from datasets import load_dataset
from transformers import ChineseCLIPModel, ChineseCLIPProcessor

os.makedirs('results', exist_ok=True)

assert torch.cuda.is_available(), 'CUDA required — install a CUDA build of PyTorch'
DEVICE = 'cuda'
print('Device:', DEVICE, torch.cuda.get_device_name(0))

LANGS = ['en', 'zh']
CLASSES = {
    'en': ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck'],
    'zh': ['飞机', '汽车', '鸟', '猫', '鹿', '狗', '青蛙', '马', '船', '卡车'],
}
TMPL = {'en': 'a photo of a {}.', 'zh': '一张{}的照片。'}
DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8
AGREE_BONUS  = 0.05   # added to conf-drop score when EN and ZH argmax agree
BLUR_RADIUS  = 12


Device: cuda NVIDIA GeForce RTX 5070 Ti


## 1. Models


In [3]:
def _clip_feat(out):
    if torch.is_tensor(out): return out
    if getattr(out, 'pooler_output', None) is not None: return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    def __init__(self):
        self.m, _, self.pp = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer('ViT-B-32')
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class ZhCLIP:
    lang = 'zh'
    def __init__(self):
        self.m = ChineseCLIPModel.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16').to(DEVICE).eval()
        self.p = ChineseCLIPProcessor.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['zh'].format(w) for w in words], padding=True, return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'], attention_mask=t['attention_mask'],
                                        token_type_ids=t.get('token_type_ids'))
        return F.normalize(_clip_feat(out), dim=-1)

def classify(model, imgs, words, batch_size=128):
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i+batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

def classify_sims(model, imgs, words, batch_size=64):
    """Return (N, C) cosine similarities."""
    all_sims = []
    tf = model.embed_texts(words)
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i+batch_size])
        all_sims.append((imf @ tf.t()).cpu().numpy())
    return np.concatenate(all_sims, axis=0)

print('Model classes defined.')


Model classes defined.


## 2. Dataset + multilingual attack


In [4]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

_sample_path = '../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json'
with open(_sample_path, encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
rows = hf.select(idx)
true = np.array(rows[label_key])
assert len(idx) == 1000 and np.array_equal(true, np.array(_saved['true']))

attack_pos = _saved['attack_pos']
assert len(attack_pos['en']) == len(idx) and len(attack_pos['l']) == len(idx)

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])]) for k in range(len(idx))])

clean_224 = [im.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
             for im in rows[image_key]]

# Tune subset: 10 per class = 100 images
tune_idx = np.concatenate([np.where(true == c)[0][:10] for c in range(10)])
print(f'Tune subset size: {len(tune_idx)}')
print(f"Attack positions: frozen from sample JSON (ref {attack_pos['ref_bw']}x{attack_pos['ref_bh']})")


Tune subset size: 100
Attack positions: frozen from sample JSON (ref 131x44)


In [5]:
import platform
_FONT_CACHE = {}

def _font_paths():
    if platform.system() == 'Windows':
        winfonts = os.path.join(os.environ.get('WINDIR', r'C:\\Windows'), 'Fonts')
        cjk   = os.path.join(winfonts, 'msyh.ttc')
        latin = os.path.join(winfonts, 'arial.ttf')
        if not os.path.exists(latin): latin = cjk
        return (cjk if os.path.exists(cjk) else None, latin if os.path.exists(latin) else None)
    for d in ['/usr/share/fonts', '/Library/Fonts', os.path.expanduser('~/.fonts')]:
        for f in ['NotoSansCJK-Regular.ttc', 'NotoSans-Regular.ttf']:
            p = os.path.join(d, f)
            if os.path.exists(p): return p, p
    return None, None

_CJK_FONT, _LAT_FONT = _font_paths()

def _get_font(fp):
    if fp not in _FONT_CACHE:
        try:    _FONT_CACHE[fp] = ImageFont.truetype(fp, FONT_SIZE) if fp else ImageFont.load_default()
        except: _FONT_CACHE[fp] = ImageFont.load_default()
    return _FONT_CACHE[fp]


def _rects_overlap(a, b):
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])

def _clamp_xy(xy, bw, bh):
    x, y = int(xy[0]), int(xy[1])
    x = max(0, min(x, max(0, DISPLAY_SIZE - bw)))
    y = max(0, min(y, max(0, DISPLAY_SIZE - bh)))
    return x, y

def draw_multilingual_attack(img, en_word, zh_word, img_idx, already_224=False):
    """Place EN at frozen slot-0, ZH at frozen slot-1 from sample JSON."""
    if not already_224:
        img = img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
    else:
        img = img.copy()
    draw = ImageDraw.Draw(img)
    xy0 = attack_pos['en'][int(img_idx)]
    xy1 = attack_pos['l'][int(img_idx)]
    for (word, fp), xy in zip([(en_word, _LAT_FONT), (zh_word, _CJK_FONT)], [xy0, xy1]):
        font = _get_font(fp)
        bb   = draw.textbbox((0, 0), word, font=font)
        bw   = (bb[2] - bb[0]) + 2 * PAD
        bh   = (bb[3] - bb[1]) + PAD + 12
        rx, ry = _clamp_xy(xy, bw, bh)
        draw.rectangle([rx, ry, rx + bw, ry + bh], fill='white')
        draw.text((rx + PAD - bb[0], ry + PAD - bb[1]), word, fill='black', font=font)
    return img

print('Building multilingual attacked images for tune subset (frozen attack_pos)...')
attacked = [draw_multilingual_attack(clean_224[i],
                                      CLASSES['en'][target[i]],
                                      CLASSES['zh'][target[i]], i, True)
            for i in tune_idx]
print(f'Built {len(attacked)} attacked images.')


Building multilingual attacked images for tune subset (frozen attack_pos)...
Built 100 attacked images.


## 3. Load models + baseline attack stats


In [6]:
models = {}
for lang, cls in [('en', EnCLIP), ('zh', ZhCLIP)]:
    t0 = time.time()
    print(f'Loading {lang}...', end=' ', flush=True)
    models[lang] = cls()
    print(f'{time.time()-t0:.1f}s')

true_tune   = true[tune_idx]
target_tune = target[tune_idx]

preds_atk = {ml: classify(models[ml], attacked, CLASSES[ml]) for ml in LANGS}
print('Attacked acc:',  {ml: f'{100*(preds_atk[ml]==true_tune).mean():.1f}%' for ml in LANGS})
print('Attacked ASR:',  {ml: f'{100*(preds_atk[ml]==target_tune).mean():.1f}%' for ml in LANGS})

# Pre-defense top class + conf per model (for conf-drop scoring)
atk_sims = {ml: classify_sims(models[ml], attacked, CLASSES[ml]) for ml in LANGS}
atk_top  = {ml: atk_sims[ml].argmax(1) for ml in LANGS}
atk_conf = {ml: atk_sims[ml].max(1) for ml in LANGS}


Loading en... 

D:\ian\2026summer\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


1.0s
Loading zh... 

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

1.4s


Attacked acc: {'en': '7.0%', 'zh': '3.0%'}
Attacked ASR: {'en': '93.0%', 'zh': '97.0%'}


## 4. Grid / occlusion / scoring helpers

- **Patches:** non-overlapping `n×n`, overlapping stride, hierarchical subdivide
- **Occlusion:** mean-fill or Gaussian blur
- **Scores:** `maxconf` (baseline) vs `confdrop` (+ agree bonus)


In [7]:
def make_grid_patches(n, size=DISPLAY_SIZE):
    ph = pw = size // n
    return [(c*pw, r*ph, (c+1)*pw, (r+1)*ph) for r in range(n) for c in range(n)]

def make_overlap_patches(patch=56, stride=28, size=DISPLAY_SIZE):
    patches = []
    for y0 in range(0, size - patch + 1, stride):
        for x0 in range(0, size - patch + 1, stride):
            patches.append((x0, y0, x0+patch, y0+patch))
    return patches

def subdivide_rect(rect, n=2):
    x0, y0, x1, y1 = rect
    rw, rh = (x1 - x0) // n, (y1 - y0) // n
    return [(x0+c*rw, y0+r*rh, x0+(c+1)*rw, y0+(r+1)*rh) for r in range(n) for c in range(n)]

def patch_center(rect):
    x0, y0, x1, y1 = rect
    return ((x0+x1)/2.0, (y0+y1)/2.0)

def patch_dist(a, b):
    ca, cb = patch_center(a), patch_center(b)
    return ((ca[0]-cb[0])**2 + (ca[1]-cb[1])**2) ** 0.5

def occlude_mean(arr, rects):
    out = arr.copy()
    fill = arr.reshape(-1, 3).mean(0).astype(np.uint8)
    for x0, y0, x1, y1 in rects:
        out[y0:y1, x0:x1] = fill
    return out

def occlude_blur(arr, rects, radius=BLUR_RADIUS):
    out = arr.copy()
    pil = Image.fromarray(arr)
    blurred = np.array(pil.filter(ImageFilter.GaussianBlur(radius=radius)))
    for x0, y0, x1, y1 in rects:
        out[y0:y1, x0:x1] = blurred[y0:y1, x0:x1]
    return out

def apply_occ(arr, rects, mode='mean'):
    return occlude_mean(arr, rects) if mode == 'mean' else occlude_blur(arr, rects)

PATCHES_4 = make_grid_patches(4)
PATCHES_6 = make_grid_patches(6)
PATCHES_OV = make_overlap_patches(56, 28)
print(f'4x4={len(PATCHES_4)}, 6x6={len(PATCHES_6)}, overlap={len(PATCHES_OV)}')


def score_maxconf(candidates):
    scores = np.zeros(len(candidates))
    for ml in LANGS:
        sims = classify_sims(models[ml], candidates, CLASSES[ml])
        scores += sims.max(1)
    return scores / len(LANGS)


def score_confdrop(candidates, img_i):
    """Mean drop of pre-defense top-class conf + agree bonus."""
    scores = np.zeros(len(candidates))
    preds = {}
    for ml in LANGS:
        sims = classify_sims(models[ml], candidates, CLASSES[ml])
        top = atk_top[ml][img_i]
        drop = atk_conf[ml][img_i] - sims[:, top]
        scores += drop
        preds[ml] = sims.argmax(1)
    scores /= len(LANGS)
    agree = (preds['en'] == preds['zh']).astype(np.float64)
    scores += AGREE_BONUS * agree
    return scores


def score_cands(candidates, img_i, scoring='maxconf'):
    if scoring == 'maxconf':
        return score_maxconf(candidates)
    return score_confdrop(candidates, img_i)

print('Helpers ready.')


4x4=16, 6x6=36, overlap=49
Helpers ready.


## 5. Defense runners


In [8]:
def run_1patch(imgs, patches, scoring='maxconf', occ='mean', label=''):
    result, best, n_fwd = [], [], 0
    for j, img in enumerate(imgs):
        arr = np.array(img.convert('RGB'))
        cands = [Image.fromarray(apply_occ(arr, [r], occ)) for r in patches]
        scores = score_cands(cands, j, scoring)
        bi = int(scores.argmax())
        best.append(bi); result.append(cands[bi])
        n_fwd += len(patches) * 2
        if (j+1) % 25 == 0:
            print(f'  {label} 1p {j+1}/{len(imgs)}')
    return result, best, n_fwd / len(imgs)


def run_2patch_greedy(imgs, patches, first_idxs, scoring='maxconf', occ='mean',
                      dist_penalty=0.0, label=''):
    """Greedy 2nd patch. If dist_penalty>0, add penalty * (1 - dist/max_dist) to discourage nearby picks."""
    result, seconds, n_fwd = [], [], 0
    max_d = (2 ** 0.5) * DISPLAY_SIZE
    for j, (img, fp) in enumerate(zip(imgs, first_idxs)):
        arr = np.array(img.convert('RGB'))
        arr1 = apply_occ(arr, [patches[fp]], occ)
        remain = [i for i in range(len(patches)) if i != fp]
        cands = [Image.fromarray(apply_occ(arr1, [patches[pi]], occ)) for pi in remain]
        scores = score_cands(cands, j, scoring)
        if dist_penalty > 0:
            for k, pi in enumerate(remain):
                d = patch_dist(patches[fp], patches[pi]) / max_d
                scores[k] -= dist_penalty * (1.0 - d)
        bi = int(scores.argmax())
        seconds.append(remain[bi]); result.append(cands[bi])
        n_fwd += len(remain) * 2
        if (j+1) % 25 == 0:
            print(f'  {label} 2p {j+1}/{len(imgs)}')
    return result, seconds, n_fwd / len(imgs)


def run_2patch_beam(imgs, patches, scoring='confdrop', occ='mean', beam=2,
                    dist_penalty=0.05, label=''):
    """Beam-k on first patch, then greedy 2nd with distance penalty; keep best pair score."""
    result, pairs, n_fwd = [], [], 0
    max_d = (2 ** 0.5) * DISPLAY_SIZE
    for j, img in enumerate(imgs):
        arr = np.array(img.convert('RGB'))
        cands1 = [Image.fromarray(apply_occ(arr, [r], occ)) for r in patches]
        s1 = score_cands(cands1, j, scoring)
        n_fwd += len(patches) * 2
        topk = list(np.argsort(-s1)[:beam])
        best_score, best_img, best_pair = -1e9, None, None
        for fp in topk:
            arr1 = apply_occ(arr, [patches[fp]], occ)
            remain = [i for i in range(len(patches)) if i != fp]
            cands2 = [Image.fromarray(apply_occ(arr1, [patches[pi]], occ)) for pi in remain]
            s2 = score_cands(cands2, j, scoring)
            n_fwd += len(remain) * 2
            if dist_penalty > 0:
                for k, pi in enumerate(remain):
                    d = patch_dist(patches[fp], patches[pi]) / max_d
                    s2[k] -= dist_penalty * (1.0 - d)
            bi = int(s2.argmax())
            if s2[bi] > best_score:
                best_score = float(s2[bi])
                best_img = cands2[bi]
                best_pair = (fp, remain[bi])
        result.append(best_img); pairs.append(best_pair)
        if (j+1) % 25 == 0:
            print(f'  {label} beam {j+1}/{len(imgs)}')
    return result, pairs, n_fwd / len(imgs)


def run_hierarchical_1p(imgs, scoring='confdrop', occ='mean', top_k=2, label=''):
    """4×4 score → subdivide top-k cells into 2×2 → pick best among coarse+fine."""
    coarse = PATCHES_4
    result, chosen, n_fwd = [], [], 0
    for j, img in enumerate(imgs):
        arr = np.array(img.convert('RGB'))
        cands = [Image.fromarray(apply_occ(arr, [r], occ)) for r in coarse]
        scores = score_cands(cands, j, scoring)
        n_fwd += len(coarse) * 2
        top = list(np.argsort(-scores)[:top_k])
        fine_rects = []
        for ti in top:
            fine_rects.extend(subdivide_rect(coarse[ti], 2))
        fine_cands = [Image.fromarray(apply_occ(arr, [r], occ)) for r in fine_rects]
        fine_scores = score_cands(fine_cands, j, scoring)
        n_fwd += len(fine_rects) * 2
        # pick global best among coarse and fine
        all_cands = cands + fine_cands
        all_scores = np.concatenate([scores, fine_scores])
        bi = int(all_scores.argmax())
        chosen.append(bi); result.append(all_cands[bi])
        if (j+1) % 25 == 0:
            print(f'  {label} hier1 {j+1}/{len(imgs)}')
    return result, chosen, n_fwd / len(imgs)


def run_hierarchical_2p(imgs, scoring='confdrop', occ='mean', top_k=2, dist_penalty=0.05, label=''):
    """Hierarchical refine for p1, then greedy 2nd over coarse 4×4 with distance penalty."""
    # Get best 1p via hierarchical, then 2nd from remaining coarse cells (+ fine of top)
    coarse = PATCHES_4
    result, pairs, n_fwd = [], [], 0
    max_d = (2 ** 0.5) * DISPLAY_SIZE
    for j, img in enumerate(imgs):
        arr = np.array(img.convert('RGB'))
        cands = [Image.fromarray(apply_occ(arr, [r], occ)) for r in coarse]
        scores = score_cands(cands, j, scoring)
        n_fwd += len(coarse) * 2
        top = list(np.argsort(-scores)[:top_k])
        fine_rects, fine_parent = [], []
        for ti in top:
            for fr in subdivide_rect(coarse[ti], 2):
                fine_rects.append(fr); fine_parent.append(ti)
        fine_cands = [Image.fromarray(apply_occ(arr, [r], occ)) for r in fine_rects]
        fine_scores = score_cands(fine_cands, j, scoring)
        n_fwd += len(fine_rects) * 2
        all_rects = list(coarse) + fine_rects
        all_scores = np.concatenate([scores, fine_scores])
        p1 = int(all_scores.argmax())
        r1 = all_rects[p1]
        arr1 = apply_occ(arr, [r1], occ)
        # second: all coarse cells that don't heavily overlap r1, plus other fine
        cand_rects = []
        for i, r in enumerate(coarse):
            if not _rects_overlap(r, r1) or i not in top:
                # allow non-overlapping coarse; also allow overlapping if not parent of r1
                if not _rects_overlap(r, r1):
                    cand_rects.append(r)
        for fr in fine_rects:
            if not _rects_overlap(fr, r1):
                cand_rects.append(fr)
        if not cand_rects:
            cand_rects = [r for r in coarse if not _rects_overlap(r, r1)] or coarse[:1]
        cands2 = [Image.fromarray(apply_occ(arr1, [r], occ)) for r in cand_rects]
        s2 = score_cands(cands2, j, scoring)
        n_fwd += len(cand_rects) * 2
        if dist_penalty > 0:
            for k, r in enumerate(cand_rects):
                d = patch_dist(r1, r) / max_d
                s2[k] -= dist_penalty * (1.0 - d)
        p2 = int(s2.argmax())
        result.append(cands2[p2]); pairs.append((p1, p2))
        if (j+1) % 25 == 0:
            print(f'  {label} hier2 {j+1}/{len(imgs)}')
    return result, pairs, n_fwd / len(imgs)

print('Runners ready.')


Runners ready.


## 6. Run ablations


In [9]:
METHODS = {}  # name -> (images, cost_per_img, seconds)

def register(name, imgs, cost, elapsed):
    METHODS[name] = {'imgs': imgs, 'cost': cost, 'time_s': elapsed}
    print(f'{name}: cost≈{cost:.0f}/img, {elapsed:.1f}s')


# --- baselines (A/B/C/D reference) ---
print('=== baseline maxconf 4x4 ===')
t0 = time.time()
imgs, best, cost = run_1patch(attacked, PATCHES_4, 'maxconf', 'mean', 'base')
register('base_1p_maxconf_mean', imgs, cost, time.time()-t0)
t0 = time.time()
imgs2, _, cost2 = run_2patch_greedy(attacked, PATCHES_4, best, 'maxconf', 'mean', 0.0, 'base')
register('base_2p_maxconf_mean', imgs2, cost + cost2, time.time()-t0)

# --- B: conf-drop on 4x4 ---
print('\n=== B confdrop 4x4 ===')
t0 = time.time()
imgs, best_cd, cost = run_1patch(attacked, PATCHES_4, 'confdrop', 'mean', 'cd4')
register('B_1p_confdrop_mean', imgs, cost, time.time()-t0)
t0 = time.time()
imgs2, _, cost2 = run_2patch_greedy(attacked, PATCHES_4, best_cd, 'confdrop', 'mean', 0.0, 'cd4')
register('B_2p_confdrop_mean', imgs2, cost + cost2, time.time()-t0)

# --- A: hierarchical ---
print('\n=== A hierarchical ===')
t0 = time.time()
imgs, _, cost = run_hierarchical_1p(attacked, 'confdrop', 'mean', 2, 'hier')
register('A_1p_hier_confdrop', imgs, cost, time.time()-t0)
t0 = time.time()
imgs2, _, cost = run_hierarchical_2p(attacked, 'confdrop', 'mean', 2, 0.05, 'hier')
register('A_2p_hier_confdrop', imgs2, cost, time.time()-t0)

# --- A: 6x6 ---
print('\n=== A 6x6 ===')
t0 = time.time()
imgs, best6, cost = run_1patch(attacked, PATCHES_6, 'confdrop', 'mean', 'g6')
register('A_1p_6x6_confdrop', imgs, cost, time.time()-t0)
t0 = time.time()
imgs2, _, cost2 = run_2patch_greedy(attacked, PATCHES_6, best6, 'confdrop', 'mean', 0.05, 'g6')
register('A_2p_6x6_confdrop', imgs2, cost + cost2, time.time()-t0)

# --- A: overlap ---
print('\n=== A overlap 56/28 ===')
t0 = time.time()
imgs, best_ov, cost = run_1patch(attacked, PATCHES_OV, 'confdrop', 'mean', 'ov')
register('A_1p_overlap_confdrop', imgs, cost, time.time()-t0)
t0 = time.time()
imgs2, _, cost2 = run_2patch_greedy(attacked, PATCHES_OV, best_ov, 'confdrop', 'mean', 0.05, 'ov')
register('A_2p_overlap_confdrop', imgs2, cost + cost2, time.time()-t0)

# --- C: blur with confdrop 4x4 ---
print('\n=== C blur ===')
t0 = time.time()
imgs, best_bl, cost = run_1patch(attacked, PATCHES_4, 'confdrop', 'blur', 'blur')
register('C_1p_confdrop_blur', imgs, cost, time.time()-t0)
t0 = time.time()
imgs2, _, cost2 = run_2patch_greedy(attacked, PATCHES_4, best_bl, 'confdrop', 'blur', 0.0, 'blur')
register('C_2p_confdrop_blur', imgs2, cost + cost2, time.time()-t0)

# --- D: beam-2 + distance on 4x4 confdrop ---
print('\n=== D beam+distance ===')
t0 = time.time()
imgs, _, cost = run_2patch_beam(attacked, PATCHES_4, 'confdrop', 'mean', beam=2, dist_penalty=0.05, label='beam')
register('D_2p_beam2_dist_confdrop', imgs, cost, time.time()-t0)

# --- combo: best geometry (6x6) + confdrop + blur + beam ---
print('\n=== combo 6x6 blur beam ===')
t0 = time.time()
imgs, _, cost = run_2patch_beam(attacked, PATCHES_6, 'confdrop', 'blur', beam=2, dist_penalty=0.05, label='combo')
register('combo_6x6_blur_beam2', imgs, cost, time.time()-t0)

print('\nAll methods done.')


=== baseline maxconf 4x4 ===


  base 1p 25/100


  base 1p 50/100


  base 1p 75/100


  base 1p 100/100
base_1p_maxconf_mean: cost≈32/img, 6.8s


...

  beam beam 100/100
D_2p_beam2_dist_confdrop: cost≈92/img, 20.0s

=== combo 6x6 blur beam ===


  combo beam 25/100


  combo beam 50/100


  combo beam 75/100


  combo beam 100/100
combo_6x6_blur_beam2: cost≈212/img, 42.0s

All methods done.


## 7. Evaluate + save


In [10]:
def eval_imgs(imgs, label, cost=None, time_s=None):
    row = {'method': label}
    for ml in LANGS:
        p = classify(models[ml], imgs, CLASSES[ml])
        row[f'acc_{ml}']  = float((p == true_tune).mean())
        row[f'asr_{ml}']  = float((p == target_tune).mean())
    row['acc_mean'] = float(np.mean([row[f'acc_{ml}'] for ml in LANGS]))
    row['asr_mean'] = float(np.mean([row[f'asr_{ml}'] for ml in LANGS]))
    if cost is not None: row['cost'] = float(cost)
    if time_s is not None: row['time_s'] = float(time_s)
    return row

rows = [eval_imgs(attacked, 'no_defense', cost=2.0, time_s=0.0)]
for name, meta in METHODS.items():
    rows.append(eval_imgs(meta['imgs'], name, meta['cost'], meta['time_s']))

rows_sorted = sorted(rows, key=lambda r: -r['acc_mean'])

print(f'\n{"Method":<32} {"EN":>7} {"ZH":>7} {"Mean":>7} {"Cost":>6} {"Time":>7}')
print('-' * 72)
for r in rows_sorted:
    cost = r.get('cost', float('nan'))
    ts   = r.get('time_s', float('nan'))
    print(f'{r["method"]:<32} {100*r["acc_en"]:>6.1f}% {100*r["acc_zh"]:>6.1f}% '
          f'{100*r["acc_mean"]:>6.1f}% {cost:>6.0f} {ts:>6.1f}s')

out = {
    'n_images': len(attacked),
    'attack': 'multilingual',
    'agree_bonus': AGREE_BONUS,
    'blur_radius': BLUR_RADIUS,
    'success_bar': {'acc_mean_min': 0.20, 'cost_max': 80},
    'rows': rows_sorted,
}
with open('results/comparison.json', 'w', encoding='utf-8') as f:
    json.dump(out, f, indent=2)
print('\nSaved results/comparison.json')

# Bar chart
fig, ax = plt.subplots(figsize=(10, max(4, 0.35*len(rows_sorted))))
names = [r['method'] for r in rows_sorted][::-1]
means = [100*r['acc_mean'] for r in rows_sorted][::-1]
colors = ['#4caf50' if m >= 20 else ('#ff9800' if m >= 15 else '#90a4ae') for m in means]
ax.barh(names, means, color=colors)
ax.axvline(20, color='green', ls='--', lw=1, label='promote bar 20%')
ax.axvline(11.7, color='gray', ls=':', lw=1, label='old grid_2p ~11.7%')
ax.set_xlabel('Mean accuracy (%)')
ax.set_title('Improved grid ablations (n=100 multilingual tune)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('results/comparison.png', dpi=140)
plt.close()
print('Saved results/comparison.png')

# Best overall vs best under cost bar
best = rows_sorted[0]
under_bar = [r for r in rows_sorted
             if r['method'] != 'no_defense' and r.get('acc_mean', 0) >= 0.20 and r.get('cost', 999) <= 80]
best_bar = under_bar[0] if under_bar else None
print(f"\nBest overall: {best['method']}  mean={100*best['acc_mean']:.1f}%  cost={best.get('cost', float('nan')):.0f}")
if best_bar:
    print(f"Best under bar (acc>=20%, cost<=80): {best_bar['method']}  "
          f"mean={100*best_bar['acc_mean']:.1f}%  cost={best_bar['cost']:.0f}")
    print('PROMOTE to n=1000: True')
else:
    print('DO NOT promote: no method clears acc>=20% at cost<=80')



Method                                EN      ZH    Mean   Cost    Time
------------------------------------------------------------------------
A_2p_overlap_confdrop              43.0%   45.0%   44.0%    194   17.2s
C_2p_confdrop_blur                 44.0%   44.0%   44.0%     62    7.3s
combo_6x6_blur_beam2               41.0%   43.0%   42.0%    212   42.0s
D_2p_beam2_dist_confdrop           42.0%   41.0%   41.5%     92   20.0s
A_2p_hier_confdrop                 42.0%   40.0%   41.0%     88   18.6s
B_2p_confdrop_mean                 40.0%   41.0%   40.5%     62    6.4s
A_1p_overlap_confdrop              36.0%   37.0%   36.5%     98   17.4s
A_2p_6x6_confdrop                  35.0%   36.0%   35.5%    142   12.7s
C_1p_confdrop_blur                 35.0%   35.0%   35.0%     32    7.5s
B_1p_confdrop_mean                 35.0%   34.0%   34.5%     32    6.8s
A_1p_hier_confdrop                 36.0%   33.0%   34.5%     48   10.7s
A_1p_6x6_confdrop                  34.0%   32.0%   33.0%     7

## 8. Phase 2 — promote best under-bar method to full n=1000

Winner from tune set under (acc ≥ 20%, cost ≤ 80): **C_2p_confdrop_blur** (and near-tied **B_2p_confdrop_mean**).
Re-run both on the full 1000 multilingual images for a fair comparison to production grid (~11.7%).


In [11]:
# Full 1000 attacked images (frozen attack_pos protocol)
print('Building full-1000 multilingual attacked images...')
attacked_1000 = [draw_multilingual_attack(clean_224[i],
                                           CLASSES['en'][target[i]],
                                           CLASSES['zh'][target[i]], i, True)
                 for i in range(len(idx))]
true_1000 = true
target_1000 = target
print(f'Built {len(attacked_1000)} images.')

# Refresh attack sims for conf-drop on full set
atk_sims = {ml: classify_sims(models[ml], attacked_1000, CLASSES[ml]) for ml in LANGS}
atk_top  = {ml: atk_sims[ml].argmax(1) for ml in LANGS}
atk_conf = {ml: atk_sims[ml].max(1) for ml in LANGS}

def eval_1000(imgs, label, cost=None, time_s=None):
    row = {'method': label, 'n': len(imgs)}
    for ml in LANGS:
        p_ = classify(models[ml], imgs, CLASSES[ml])
        row[f'acc_{ml}'] = float((p_ == true_1000).mean())
        row[f'asr_{ml}'] = float((p_ == target_1000).mean())
    row['acc_mean'] = float(np.mean([row[f'acc_{ml}'] for ml in LANGS]))
    row['asr_mean'] = float(np.mean([row[f'asr_{ml}'] for ml in LANGS]))
    if cost is not None: row['cost'] = float(cost)
    if time_s is not None: row['time_s'] = float(time_s)
    return row

phase2 = {}

print('\n=== no_defense n=1000 ===')
phase2['no_defense'] = eval_1000(attacked_1000, 'no_defense', 2.0, 0.0)
print(phase2['no_defense'])

print('\n=== baseline 2p maxconf mean (old grid) ===')
t0 = time.time()
imgs, best, cost1 = run_1patch(attacked_1000, PATCHES_4, 'maxconf', 'mean', 'base1k')
imgs2, _, cost2 = run_2patch_greedy(attacked_1000, PATCHES_4, best, 'maxconf', 'mean', 0.0, 'base1k')
phase2['base_2p_maxconf_mean'] = eval_1000(imgs2, 'base_2p_maxconf_mean', cost1+cost2, time.time()-t0)
print(phase2['base_2p_maxconf_mean'])

print('\n=== B 2p confdrop mean ===')
t0 = time.time()
imgs, best, cost1 = run_1patch(attacked_1000, PATCHES_4, 'confdrop', 'mean', 'B1k')
imgs2, _, cost2 = run_2patch_greedy(attacked_1000, PATCHES_4, best, 'confdrop', 'mean', 0.0, 'B1k')
phase2['B_2p_confdrop_mean'] = eval_1000(imgs2, 'B_2p_confdrop_mean', cost1+cost2, time.time()-t0)
print(phase2['B_2p_confdrop_mean'])

print('\n=== C 2p confdrop blur (promote winner) ===')
t0 = time.time()
imgs, best, cost1 = run_1patch(attacked_1000, PATCHES_4, 'confdrop', 'blur', 'C1k')
imgs2, _, cost2 = run_2patch_greedy(attacked_1000, PATCHES_4, best, 'confdrop', 'blur', 0.0, 'C1k')
phase2['C_2p_confdrop_blur'] = eval_1000(imgs2, 'C_2p_confdrop_blur', cost1+cost2, time.time()-t0)
print(phase2['C_2p_confdrop_blur'])

rows_1000 = list(phase2.values())
rows_1000_sorted = sorted(rows_1000, key=lambda r: -r['acc_mean'])
print(f'\n{"Method":<28} {"EN":>7} {"ZH":>7} {"Mean":>7} {"Cost":>6}')
print('-' * 60)
for r in rows_1000_sorted:
    print(f'{r["method"]:<28} {100*r["acc_en"]:>6.1f}% {100*r["acc_zh"]:>6.1f}% '
          f'{100*r["acc_mean"]:>6.1f}% {r.get("cost", float("nan")):>6.0f}')

with open('results/comparison_n1000.json', 'w', encoding='utf-8') as f:
    json.dump({'n_images': 1000, 'attack': 'multilingual', 'rows': rows_1000_sorted}, f, indent=2)
print('Saved results/comparison_n1000.json')


Building full-1000 multilingual attacked images...
Built 1000 images.



=== no_defense n=1000 ===


{'method': 'no_defense', 'n': 1000, 'acc_en': 0.045, 'asr_en': 0.953, 'acc_zh': 0.064, 'asr_zh': 0.936, 'acc_mean': 0.0545, 'asr_mean': 0.9445, 'cost': 2.0, 'time_s': 0.0}

=== baseline 2p maxconf mean (old grid) ===


  base1k 1p 25/1000


  base1k 1p 50/1000


...

  C1k 2p 925/1000


  C1k 2p 950/1000


  C1k 2p 975/1000


  C1k 2p 1000/1000


{'method': 'C_2p_confdrop_blur', 'n': 1000, 'acc_en': 0.478, 'asr_en': 0.445, 'acc_zh': 0.492, 'asr_zh': 0.451, 'acc_mean': 0.485, 'asr_mean': 0.448, 'cost': 62.0, 'time_s': 143.32873058319092}

Method                            EN      ZH    Mean   Cost
------------------------------------------------------------
C_2p_confdrop_blur             47.8%   49.2%   48.5%     62
B_2p_confdrop_mean             44.9%   46.5%   45.7%     62
base_2p_maxconf_mean            5.4%   14.8%   10.1%     62
no_defense                      4.5%    6.4%    5.5%      2
Saved results/comparison_n1000.json


## 9. Visualize — conf-drop vs maxconf, mean vs blur

**Conf-drop:** score each candidate by how much the *pre-defense top class* confidence falls (avg EN+ZH), plus a small bonus if EN/ZH agree after occlusion.

**Mean fill:** replace the chosen patch with the image’s average RGB (flat block).

**Blur fill:** Gaussian-blur only inside the chosen patch — destroys text strokes, keeps more object structure.

Red/blue/green boxes mark the two patches the search selected.


In [12]:
# Rebuild defended images for the tune set (fast enough) and save example panels.
print('Building viz defences on tune set...')
t0 = time.time()
viz_old, pairs_old = [], []
viz_mean, pairs_mean = [], []
viz_blur, pairs_blur = [], []

# reuse runners
imgs, best, _ = run_1patch(attacked, PATCHES_4, 'maxconf', 'mean', 'viz_old')
imgs2, secs, _ = run_2patch_greedy(attacked, PATCHES_4, best, 'maxconf', 'mean', 0.0, 'viz_old')
viz_old, pairs_old = imgs2, list(zip(best, secs))

imgs, best, _ = run_1patch(attacked, PATCHES_4, 'confdrop', 'mean', 'viz_mean')
imgs2, secs, _ = run_2patch_greedy(attacked, PATCHES_4, best, 'confdrop', 'mean', 0.0, 'viz_mean')
viz_mean, pairs_mean = imgs2, list(zip(best, secs))

imgs, best, _ = run_1patch(attacked, PATCHES_4, 'confdrop', 'blur', 'viz_blur')
imgs2, secs, _ = run_2patch_greedy(attacked, PATCHES_4, best, 'confdrop', 'blur', 0.0, 'viz_blur')
viz_blur, pairs_blur = imgs2, list(zip(best, secs))
print(f'Done in {time.time()-t0:.1f}s')

preds_atk_v = {ml: classify(models[ml], attacked, CLASSES[ml]) for ml in LANGS}
preds_old_v = {ml: classify(models[ml], viz_old, CLASSES[ml]) for ml in LANGS}
preds_mean_v = {ml: classify(models[ml], viz_mean, CLASSES[ml]) for ml in LANGS}
preds_blur_v = {ml: classify(models[ml], viz_blur, CLASSES[ml]) for ml in LANGS}

good = [i for i in range(len(attacked))
        if preds_atk_v['en'][i] != true_tune[i] and preds_blur_v['en'][i] == true_tune[i]]
show = (good[:6] if len(good) >= 6 else list(range(6)))

def _draw_patches(img, pair, color=(255, 0, 0)):
    im = img.copy(); d = ImageDraw.Draw(im)
    for pi in pair:
        x0, y0, x1, y1 = PATCHES_4[pi]
        d.rectangle([x0, y0, x1-1, y1-1], outline=color, width=3)
    return im

cols = ['Attacked', 'Old maxconf+mean', 'Conf-drop+mean', 'Conf-drop+blur']
fig, axes = plt.subplots(len(show), 4, figsize=(12, 3.1 * len(show)))
for row, i in enumerate(show):
    panels = [
        attacked[i],
        _draw_patches(viz_old[i], pairs_old[i], (200, 40, 40)),
        _draw_patches(viz_mean[i], pairs_mean[i], (40, 120, 200)),
        _draw_patches(viz_blur[i], pairs_blur[i], (40, 160, 80)),
    ]
    pred_sets = [preds_atk_v, preds_old_v, preds_mean_v, preds_blur_v]
    for col, (ax, im, ps) in enumerate(zip(axes[row], panels, pred_sets)):
        ax.imshow(im); ax.axis('off')
        # EN class names only in titles (default matplotlib font lacks CJK glyphs)
        en_name = CLASSES['en'][ps['en'][i]]
        zh_as_en = CLASSES['en'][ps['zh'][i]]
        en_ok = 'OK' if ps['en'][i] == true_tune[i] else 'X'
        zh_ok = 'OK' if ps['zh'][i] == true_tune[i] else 'X'
        title = f"{cols[col]}\nEN {en_name} [{en_ok}] | ZH->{zh_as_en} [{zh_ok}]"
        if col == 0:
            title = (f"true={CLASSES['en'][true_tune[i]]} tgt={CLASSES['en'][target_tune[i]]}\n" + title)
        ax.set_title(title, fontsize=8)
plt.suptitle('Grid defense examples — boxes mark selected 4x4 patches', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('results/defence_examples.png', dpi=140, bbox_inches='tight')
plt.close()
print('Saved results/defence_examples.png')

# Fill close-up
i = show[0]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for ax, im, title in zip(
    axes,
    [attacked[i], viz_old[i], viz_mean[i], viz_blur[i]],
    ['Attacked', 'Maxconf + mean', 'Conf-drop + mean', 'Conf-drop + blur'],
):
    ax.imshow(im); ax.axis('off'); ax.set_title(title, fontsize=9)
plt.suptitle('Mean fill = flat average color; blur fill = Gaussian blur inside patches', fontsize=11)
plt.tight_layout()
plt.savefig('results/fill_comparison.png', dpi=140, bbox_inches='tight')
plt.close()
print('Saved results/fill_comparison.png')

# n=1000 bars from saved JSON
with open('results/comparison_n1000.json', encoding='utf-8') as f:
    n1000 = json.load(f)
rows_b = sorted(n1000['rows'], key=lambda r: r['acc_mean'])
fig, ax = plt.subplots(figsize=(8, 3.5))
names = [r['method'] for r in rows_b]
means = [100 * r['acc_mean'] for r in rows_b]
colors = ['#90a4ae' if ('no_defense' in n or n.startswith('base_')) else '#4caf50' for n in names]
ax.barh(names, means, color=colors)
ax.axvline(33.2, color='#ff9800', ls='--', lw=1, label='cam_2mod ~33%')
ax.axvline(72.6, color='#7e57c2', ls='--', lw=1, label='Attn-last ~72.6%')
ax.set_xlabel('Mean accuracy (%)')
ax.set_title('Improved grid — full multilingual n=1000')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('results/n1000_bars.png', dpi=140)
plt.close()
print('Saved results/n1000_bars.png')


Building viz defences on tune set...


  viz_old 1p 25/100


  viz_old 1p 50/100


  viz_old 1p 75/100


  viz_old 1p 100/100


...

  viz_blur 2p 50/100


  viz_blur 2p 75/100


  viz_blur 2p 100/100
Done in 40.6s


Saved results/defence_examples.png


Saved results/fill_comparison.png
Saved results/n1000_bars.png
